# ДЗ №3. Дообучение YOLO11n на домашних классах

Ноутбук готовит подвыборку COCO train2017 по 10 домашним классам, дообучает YOLO11n
и экспортирует веса в ONNX для трёх размеров входа. Оценка моделей вынесена
в `03_evaluate_models.ipynb`, чтобы обучение и оценка помещались в сессию Kaggle.

**Настройки Kaggle:** Accelerator — GPU T4, Internet — On.
**Входные данные:** датасет COCO 2017 с `train2017` и `instances_train2017.json`.

In [ ]:
import subprocess
import sys

# Ветка или коммит репозитория с пакетом photoquest_eval.
REPO_REF = "hw_3"
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"photoquest-eval[yolo] @ git+https://github.com/plekhovrodion/mlsd@{REPO_REF}",
    ]
)
REPO_COMMIT = subprocess.check_output(
    ["git", "ls-remote", "https://github.com/plekhovrodion/mlsd", REPO_REF], text=True
).split()[0]
print("photoquest_eval:", REPO_REF, REPO_COMMIT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import shutil  # noqa: F401 — ячейка общая для обоих ноутбуков
import time
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from photoquest_eval.config import (  # noqa: F401 — ячейка общая для обоих ноутбуков
    HOME_CLASSES,
    IMAGE_SIZES,
    SEED,
    TARGETS,
    YOLO_SETTINGS,
)

STARTED_UTC = datetime.now(UTC).isoformat()
STARTED = time.perf_counter()
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle Settings → Accelerator → GPU T4")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_versions(*names):
    return {name: importlib.metadata.version(name) for name in names}

In [ ]:
from ultralytics import YOLO

from photoquest_eval.data import coco_to_yolo, write_yolo_yaml

OUTPUT = Path("/kaggle/working/hw3_train")
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = "yolo11n_home"

# Гиперпараметры фиксируются до обучения и сохраняются в metadata.json.
TRAIN_SETTINGS = {
    "base_model": "yolo11n.pt",
    "epochs": 30,
    "patience": 10,
    "imgsz": 640,
    "batch": 32,
    "optimizer": "auto",
    "seed": SEED,
    "deterministic": True,
    "workers": 4,
    "save_period": 5,
}
# Фото копируются на локальный диск и уменьшаются до размера входа: чтение из
# /kaggle/input идёт с сетевого хранилища и замедляет каждую эпоху в разы.
DATA_SETTINGS = {
    "negative_fraction": 0.1,
    "val_fraction": 0.1,
    "seed": SEED,
    "max_side": TRAIN_SETTINGS["imgsz"],
}

## Данные

Ищем аннотации COCO train2017 во входных данных ноутбука.

In [ ]:
paths = sorted(Path("/kaggle/input").rglob("instances_train2017.json"))
if not paths:
    raise FileNotFoundError(
        "Добавьте во входные данные датасет COCO 2017 с train2017 "
        "(Add Input → Datasets → coco 2017)"
    )
TRAIN_ANNOTATIONS = paths[0]
coco_root = TRAIN_ANNOTATIONS.parent.parent
TRAIN_IMAGES = next(
    p for p in (coco_root / "train2017", coco_root / "images" / "train2017") if p.is_dir()
)
annotation_hash = sha256(TRAIN_ANNOTATIONS)
print(TRAIN_ANNOTATIONS, annotation_hash)
print(TRAIN_IMAGES, sum(1 for _ in TRAIN_IMAGES.glob("*.jpg")), "JPEG")

In [ ]:
# /tmp — локальный диск сессии; он не попадает в вывод ноутбука и не занимает
# лимит /kaggle/working. Повторный запуск пропускает уже скопированные фото.
DATASET = Path("/tmp/coco_home_yolo")
copy_started = time.perf_counter()
split_stats = coco_to_yolo(TRAIN_ANNOTATIONS, TRAIN_IMAGES, DATASET, TARGETS, **DATA_SETTINGS)
copy_seconds = time.perf_counter() - copy_started
print(f"Подготовка выборки заняла {copy_seconds / 60:.1f} мин")
DATASET_YAML = write_yolo_yaml(DATASET, TARGETS)
display(pd.DataFrame(split_stats).T)
print(DATASET_YAML.read_text())

## Обучение

In [ ]:
CHECKPOINTS = {
    "yolov8n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt",
        "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36",
    ),
    "yolo11n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt",
        "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
    ),
}


def get_checkpoint(filename: str, folder: Path) -> Path:
    """Скачивает предобученные веса и сверяет SHA-256 с прогоном ДЗ №2."""
    url, expected = CHECKPOINTS[filename]
    path = folder / filename
    folder.mkdir(parents=True, exist_ok=True)
    if not path.is_file():
        urllib.request.urlretrieve(url, path)
    if sha256(path) != expected:
        raise ValueError(f"Не совпал SHA-256 весов {filename}")
    return path

In [ ]:
base = get_checkpoint(TRAIN_SETTINGS["base_model"], OUTPUT / "pretrained")
model = YOLO(str(base))
train_started = time.perf_counter()
results = model.train(
    data=str(DATASET_YAML),
    project=str(OUTPUT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    **{k: v for k, v in TRAIN_SETTINGS.items() if k != "base_model"},
)
train_seconds = time.perf_counter() - train_started
RUN = OUTPUT / RUN_NAME
BEST = RUN / "weights" / "best.pt"
print(f"Обучение заняло {train_seconds / 3600:.2f} ч, лучшие веса: {BEST}")

## Кривые обучения

Функция потерь Ultralytics для детекции складывается из трёх частей: `box` — ошибка
положения рамки (CIoU), `cls` — ошибка класса (бинарная кросс-энтропия),
`dfl` — distribution focal loss для границ рамки.

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(RUN / "results.csv")
history.columns = history.columns.str.strip()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for axis, part in zip(axes[:3], ("box", "cls", "dfl"), strict=True):
    axis.plot(history["epoch"], history[f"train/{part}_loss"], label="train")
    axis.plot(history["epoch"], history[f"val/{part}_loss"], label="val")
    axis.set(title=f"{part}_loss", xlabel="эпоха")
    axis.legend()
axes[3].plot(history["epoch"], history["metrics/mAP50(B)"], label="mAP@0.50")
axes[3].plot(history["epoch"], history["metrics/mAP50-95(B)"], label="mAP@0.50:0.95")
axes[3].set(title="Качество на val", xlabel="эпоха")
axes[3].legend()
fig.tight_layout()
fig.savefig(OUTPUT / "training_curves.png", dpi=150)
plt.show()
history.to_csv(OUTPUT / "training_history.csv", index=False)
display(history.tail(3))

## Экспорт в ONNX

Для браузера в ДЗ №4 готовим файлы для трёх размеров входа.

In [ ]:
EXPORTS = OUTPUT / "export"
EXPORTS.mkdir(exist_ok=True)
shutil.copy(BEST, EXPORTS / f"{RUN_NAME}.pt")
export_rows = []
for size in IMAGE_SIZES:
    exported = Path(YOLO(str(BEST)).export(format="onnx", imgsz=size, simplify=True))
    target = EXPORTS / f"{RUN_NAME}_{size}.onnx"
    shutil.move(exported, target)
    export_rows.append(
        {
            "file": target.name,
            "imgsz": size,
            "MiB": target.stat().st_size / 2**20,
            "sha256": sha256(target),
        }
    )
export_rows.append(
    {
        "file": f"{RUN_NAME}.pt",
        "imgsz": 640,
        "MiB": (EXPORTS / f"{RUN_NAME}.pt").stat().st_size / 2**20,
        "sha256": sha256(EXPORTS / f"{RUN_NAME}.pt"),
    }
)
exports = pd.DataFrame(export_rows)
exports.to_csv(OUTPUT / "exports.csv", index=False)
display(exports)

In [ ]:
metadata = {
    "repo_ref": REPO_REF,
    "repo_commit": REPO_COMMIT,
    "train_annotations_sha256": annotation_hash,
    "targets": list(TARGETS),
    "openimages_names": HOME_CLASSES,
    "data_settings": DATA_SETTINGS,
    "split_stats": split_stats,
    "train_settings": TRAIN_SETTINGS,
    "best_epoch": int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"]),
    "epochs_run": int(history["epoch"].max()),
    "data_prep_seconds": round(copy_seconds, 1),
    "train_seconds": round(train_seconds, 1),
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "packages": package_versions(
        "torch", "ultralytics", "numpy", "onnx", "onnxruntime", "onnxslim"
    ),
    "started_utc": STARTED_UTC,
    "elapsed_seconds": round(time.perf_counter() - STARTED, 1),
}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
# В архив для репозитория идут метрики, графики и экспорт; промежуточные чекпоинты — нет.
with zipfile.ZipFile("/kaggle/working/hw3_train_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.rglob("*")):
        relative = path.relative_to(OUTPUT)
        skipped = {"pretrained", "weights"} & set(relative.parts)
        if path.is_file() and not skipped:
            z.write(path, relative)
print(json.dumps(metadata, indent=2, ensure_ascii=False))